# Chapter 16: Domain-Specific RL: Code, Math, Tools, Dialogue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part3_advanced/16_domain_specific_rl.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar & Michael Chertushkin (Packt, 2025)  
> **Chapter 16:** Domain-Specific RL: Code, Math, Tools, Dialogue  
> **Notebook:** `part3_advanced/16_domain_specific_rl.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 16 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers==4.40.0 torch accelerate

## 1. Imports


In [ ]:
import re
import os
import sys
import math
import random
import subprocess
import tempfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Load Qwen/Qwen2.5-0.5B


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print(f'Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')


def gen(prompt: str, max_new: int = 80, temp: float = 0.8) -> str:
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=150).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, temperature=temp,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    new = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

## 3. Domain 1 — Code Reward

### 3a. Code Reward Function


In [ ]:
def code_reward(generated_code: str, test_cases: list, timeout: int = 5) -> dict:
    """
    Execute `generated_code` and run each test case string.
    Returns {'passed': int, 'total': int, 'reward': float, 'errors': list}.
    """
    passed = 0
    errors = []
    for test in test_cases:
        combined = generated_code.strip() + '\n\n' + test.strip()
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(combined)
            fname = f.name
        try:
            result = subprocess.run(
                [sys.executable, fname],
                capture_output=True, text=True, timeout=timeout
            )
            if result.returncode == 0:
                passed += 1
            else:
                errors.append(result.stderr.strip()[:100])
        except subprocess.TimeoutExpired:
            errors.append('TIMEOUT')
        finally:
            os.unlink(fname)
    reward = passed / len(test_cases) if test_cases else 0.0
    return {'passed': passed, 'total': len(test_cases), 'reward': reward, 'errors': errors}


print('code_reward defined.')

### 3b. Five Code Problems


In [ ]:
CODE_TASKS = [
    {
        'name': 'reverse_string',
        'prompt': 'Write a Python function called reverse_string(s) that returns the reversed string.',
        'reference': 'def reverse_string(s):\n    return s[::-1]',
        'tests': [
            'assert reverse_string("hello") == "olleh", "failed"\nprint("ok")',
            'assert reverse_string("") == "", "failed"\nprint("ok")',
        ],
    },
    {
        'name': 'sum_list',
        'prompt': 'Write a Python function called sum_list(lst) that returns the sum of all elements.',
        'reference': 'def sum_list(lst):\n    return sum(lst)',
        'tests': [
            'assert sum_list([1,2,3]) == 6\nprint("ok")',
            'assert sum_list([]) == 0\nprint("ok")',
        ],
    },
    {
        'name': 'find_max',
        'prompt': 'Write a Python function called find_max(lst) that returns the maximum element.',
        'reference': 'def find_max(lst):\n    return max(lst)',
        'tests': [
            'assert find_max([3,1,4,1,5]) == 5\nprint("ok")',
            'assert find_max([-1,-5,-2]) == -1\nprint("ok")',
        ],
    },
    {
        'name': 'count_vowels',
        'prompt': 'Write a Python function called count_vowels(s) that counts vowels (a,e,i,o,u) in a string.',
        'reference': 'def count_vowels(s):\n    return sum(1 for c in s.lower() if c in "aeiou")',
        'tests': [
            'assert count_vowels("hello") == 2\nprint("ok")',
            'assert count_vowels("xyz") == 0\nprint("ok")',
        ],
    },
    {
        'name': 'fibonacci',
        'prompt': 'Write a Python function called fibonacci(n) that returns the nth Fibonacci number (0-indexed, fib(0)=0, fib(1)=1).',
        'reference': 'def fibonacci(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a+b\n    return a',
        'tests': [
            'assert fibonacci(0) == 0\nassert fibonacci(1) == 1\nprint("ok")',
            'assert fibonacci(7) == 13\nprint("ok")',
        ],
    },
]

print(f'{len(CODE_TASKS)} code tasks defined.')

### 3c. Generate Code with Qwen/Qwen2.5-0.5B and Score


In [ ]:
def extract_python_code(text: str) -> str:
    """Try to extract a function definition from generated text."""
    m = re.search(r'(def \w+\(.*?\n(?:[ \t]+.+\n?)*)', text)
    if m:
        return m.group(1)
    return text  # return as-is and let the verifier handle the error


results = []
print(f'{"Task":<18} {"Model reward":>13}  {"Reference reward":>16}')
print('-' * 55)

for task in CODE_TASKS:
    prompt = f'# Python\n# {task["prompt"]}\n'
    raw_code = gen(prompt, max_new=80, temp=0.7)
    model_code = extract_python_code(raw_code)

    model_r     = code_reward(model_code,        task['tests'])
    reference_r = code_reward(task['reference'], task['tests'])

    results.append({'task': task['name'], 'model': model_r['reward'], 'ref': reference_r['reward']})
    print(f'{task["name"]:<18} {model_r["reward"]:>7.2f} ({model_r["passed"]}/{model_r["total"]})  '
          f'{reference_r["reward"]:>7.2f} ({reference_r["passed"]}/{reference_r["total"]})')

### 3d. Reward Distribution Across Generations


In [ ]:
# Generate 15 candidates for the reverse_string task; plot reward histogram
task0 = CODE_TASKS[0]
prompt0 = f'# Python\n# {task0["prompt"]}\n'
sample_rewards = []
for _ in range(15):
    raw = gen(prompt0, max_new=60, temp=random.uniform(0.5, 1.2))
    code = extract_python_code(raw)
    r = code_reward(code, task0['tests'])['reward']
    sample_rewards.append(r)

model_rewards  = [r['model'] for r in results]
ref_rewards    = [r['ref']   for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sample_rewards, bins=[0, 0.25, 0.5, 0.75, 1.01], edgecolor='black', color='steelblue')
axes[0].set_xlabel('Code reward'); axes[0].set_ylabel('Count')
axes[0].set_title(f'Code Reward Distribution\n(15 generations, {task0["name"]})')
axes[0].grid(True, alpha=0.3)

x = np.arange(len(CODE_TASKS))
axes[1].bar(x - 0.2, model_rewards, 0.4, label='Qwen/Qwen2.5-0.5B', color='salmon')
axes[1].bar(x + 0.2, ref_rewards,   0.4, label='Reference',  color='steelblue')
axes[1].set_xticks(x); axes[1].set_xticklabels([r['task'] for r in results], rotation=20, ha='right')
axes[1].set_ylim(0, 1.2); axes[1].set_ylabel('Reward'); axes[1].legend()
axes[1].set_title('Code Reward: Model vs Reference')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('code_rewards.png', dpi=120)
plt.show()

## 4. Domain 2 — Math Reward

### 4a. Multi-Format Answer Extraction


In [ ]:
def math_extract(text: str):
    """
    Robust answer extraction supporting multiple formats:
    'The answer is 42', '42', 'x = 42', '= 42', 'answer: 42'
    Returns float or None.
    """
    patterns = [
        r'(?:the\s+)?(?:answer|result|value)\s+(?:is|=|:)\s*(-?\d+\.?\d*)',
        r'[a-z]\s*=\s*(-?\d+\.?\d*)',
        r'=\s*(-?\d+\.?\d*)\s*$',
        r'^\s*(-?\d+\.?\d*)\s*$',
        r'(-?\d+\.?\d*)\s*$',
        r'(-?\d+\.?\d*)',
    ]
    for pat in patterns:
        m = re.search(pat, text.strip(), re.IGNORECASE | re.MULTILINE)
        if m:
            try: return float(m.group(1))
            except ValueError: pass
    return None


def math_reward(response: str, gold: float, tol: float = 0.5) -> float:
    pred = math_extract(response)
    return 1.0 if (pred is not None and abs(pred - gold) <= tol) else 0.0


# Test multi-format extraction
format_tests = [
    ('The answer is 42', 42),
    ('42', 42),
    ('x = 42', 42),
    ('= 42', 42),
    ('result: 3.14', 3.14),
    ('So we get -7', -7),
]
print('Multi-format extraction tests:')
for text, expected in format_tests:
    pred = math_extract(text)
    ok = 'OK  ' if pred == expected else 'FAIL'
    print(f'  [{ok}] {text:30s} -> {pred}  (expected {expected})')

### 4b. Test on 10 Arithmetic Problems


In [ ]:
MATH_PROBLEMS = [
    ('What is 3 + 4?', 7),
    ('What is 10 - 3?', 7),
    ('What is 5 + 6?', 11),
    ('What is 8 - 2?', 6),
    ('What is 4 + 9?', 13),
    ('What is 15 - 7?', 8),
    ('What is 2 + 11?', 13),
    ('What is 20 - 5?', 15),
    ('What is 7 + 8?', 15),
    ('What is 9 - 4?', 5),
]

math_results = []
for question, gold in MATH_PROBLEMS:
    prompt = f'Q: {question}\nA: The answer is'
    resp = gen(prompt, max_new=20, temp=0.5)
    r = math_reward(resp, gold)
    pred = math_extract(resp)
    math_results.append(r)
    tag = 'CORRECT' if r else 'WRONG  '
    print(f'[{tag}] gold={gold:4d}  pred={str(pred):6s}  | {resp[:50]}')

print(f'\nMath reward accuracy: {sum(math_results)}/{len(math_results)} = {np.mean(math_results):.1%}')

## 5. Domain 3 — Tool-Use Reward

### 5a. Tool-Calling Simulation


In [ ]:
TOOL_REGISTRY = {
    'calculator': lambda expr: str(eval(expr, {'__builtins__': {}}, {})),  # safe arithmetic only
    'dictionary': lambda word: {'entropy': 'Measure of disorder or uncertainty in a system.',
                                'recursion': 'A function that calls itself.',
                                'gradient': 'Rate of change of a function.'}.get(word.lower(), 'Not found.'),
    'weather':    lambda city: f'Mocked: {city} is currently 22°C and sunny.',
}


def tool_use_reward(actions: list, task_correct: bool,
                    efficiency_weight: float = 0.1,
                    error_penalty: float = 0.5) -> dict:
    """
    actions: list of {'tool': str, 'arg': str, 'result': str, 'error': bool}
    """
    task_r     = 1.0 if task_correct else 0.0
    eff_r      = -efficiency_weight * len(actions)
    error_r    = -error_penalty * sum(1 for a in actions if a.get('error', False))
    total      = task_r + eff_r + error_r
    return {'task': task_r, 'efficiency': eff_r, 'error': error_r, 'total': total}


def simulate_agent(task: str, plan: list) -> list:
    """Execute a list of (tool, arg) calls; return action dicts with results."""
    actions = []
    for tool_name, arg in plan:
        if tool_name not in TOOL_REGISTRY:
            actions.append({'tool': tool_name, 'arg': arg, 'result': 'ERROR: unknown tool', 'error': True})
        else:
            try:
                result = TOOL_REGISTRY[tool_name](arg)
                actions.append({'tool': tool_name, 'arg': arg, 'result': result, 'error': False})
            except Exception as e:
                actions.append({'tool': tool_name, 'arg': arg, 'result': str(e), 'error': True})
    return actions


TOOL_SCENARIOS = [
    {
        'task': 'What is 15 * 4?',
        'expected': '60',
        'plans': [
            [('calculator', '15 * 4')],                           # optimal
            [('dictionary', '15'), ('calculator', '15 * 4')],     # wasteful
            [('invalid_tool', '15 * 4')],                         # error
        ],
    },
    {
        'task': 'Define the word entropy',
        'expected': 'Measure of disorder',
        'plans': [
            [('dictionary', 'entropy')],                           # optimal
            [('weather', 'entropy'), ('dictionary', 'entropy')],   # wasteful
        ],
    },
]

print('Tool-use reward simulation:')
for scenario in TOOL_SCENARIOS:
    print(f'\nTask: {scenario["task"]}')
    for i, plan in enumerate(scenario['plans']):
        actions = simulate_agent(scenario['task'], plan)
        last_result = actions[-1]['result'] if actions else ''
        correct = scenario['expected'].lower() in last_result.lower()
        r = tool_use_reward(actions, task_correct=correct)
        print(f'  Plan {i+1}: {[p[0] for p in plan]}')
        print(f'    task={r["task"]:.1f} efficiency={r["efficiency"]:.2f} error={r["error"]:.2f} total={r["total"]:.2f}')

## 6. Domain 4 — Dialogue Reward

### 6a. Dialogue Quality Metrics


In [ ]:
def simple_word_overlap(a: str, b: str) -> float:
    """Jaccard similarity between word sets — proxy for coherence."""
    words_a = set(a.lower().split())
    words_b = set(b.lower().split())
    if not words_a or not words_b:
        return 0.0
    return len(words_a & words_b) / len(words_a | words_b)


UNSAFE_WORDS = ['kill', 'attack', 'harm', 'bomb', 'illegal', 'steal', 'weapon']


def dialogue_reward(query: str, response: str) -> dict:
    """
    Three-component dialogue reward:
    - coherence: word-overlap between query and response
    - engagement: response length relative to query length
    - safety: 1 if no unsafe words, 0 otherwise
    """
    coherence  = simple_word_overlap(query, response)
    q_len = max(len(query.split()), 1)
    r_len = len(response.split())
    engagement = min(r_len / (q_len * 3), 1.0)  # normalise to [0,1]
    safety     = 0.0 if any(w in response.lower() for w in UNSAFE_WORDS) else 1.0
    total = 0.4 * coherence + 0.3 * engagement + 0.3 * safety
    return {'coherence': coherence, 'engagement': engagement, 'safety': safety, 'total': total}


DIALOGUES = [
    (
        'How does photosynthesis work?',
        'Photosynthesis is the process by which plants convert light energy into chemical energy, '
        'producing glucose and oxygen from carbon dioxide and water.',
    ),
    (
        'What is machine learning?',
        'OK.',  # too short, low engagement
    ),
    (
        'Tell me about healthy eating.',
        'A balanced diet includes fruits, vegetables, whole grains, and lean proteins. '
        'Eating a variety of foods helps ensure you get all essential nutrients.',
    ),
    (
        'How do I deal with stress?',
        'You should kill the source of stress immediately with aggressive attack.',  # unsafe
    ),
    (
        'What are the benefits of exercise?',
        'Exercise helps reduce stress, improve mood, strengthen muscles, and lower the risk '
        'of chronic diseases. Even 30 minutes of walking a day can have significant health benefits.',
    ),
]

print(f'{"#":<3} {"Coherence":>10} {"Engagement":>11} {"Safety":>7} {"Total":>7}')
print('-' * 45)
dial_results = []
for i, (q, r) in enumerate(DIALOGUES, 1):
    dr = dialogue_reward(q, r)
    dial_results.append(dr)
    print(f'{i:<3} {dr["coherence"]:>10.3f} {dr["engagement"]:>11.3f} {dr["safety"]:>7.1f} {dr["total"]:>7.3f}')

## 7. Automated vs Human Labels — Domain Comparison


In [ ]:
domain_comparison = {
    'Domain':          ['Code',        'Math',        'Tool-use',       'Dialogue'],
    'Reward source':   ['Subprocess',  'Regex+eval',  'Env simulation', 'Heuristics'],
    'Cost vs human':   [0.001,         0.001,         0.01,             0.05],    # relative
    'Speed (rel)':     [100,           10000,         1000,             500],
    'Reliability':     [0.99,          0.95,          0.90,             0.70],    # approx
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
domains = domain_comparison['Domain']
colors  = ['#4a90d9', '#7dc67d', '#f5a623', '#e8635a']

axes[0].bar(domains, domain_comparison['Cost vs human'], color=colors)
axes[0].set_title('Cost vs Human Labelling (relative)'); axes[0].set_ylabel('Relative cost')
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].bar(domains, domain_comparison['Speed (rel)'], color=colors)
axes[1].set_title('Speed vs Human Labelling (relative)'); axes[1].set_ylabel('Speed factor')
axes[1].set_yscale('log'); axes[1].grid(True, axis='y', alpha=0.3)

axes[2].bar(domains, domain_comparison['Reliability'], color=colors)
axes[2].set_ylim(0, 1.1); axes[2].set_ylabel('Reliability')
axes[2].set_title('Signal Reliability'); axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('domain_comparison.png', dpi=120)
plt.show()

## 8. Reward Engineering Best Practices


In [ ]:
# Demonstrate normalisation and clipping
raw_rewards = np.array([0.0, 0.1, 0.5, 0.8, 1.0, 2.5, -0.3, 0.95, 0.72, 1.1])


def normalise_rewards(rewards: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Z-score normalise rewards within a batch."""
    return (rewards - rewards.mean()) / (rewards.std() + eps)


def clip_rewards(rewards: np.ndarray, low: float = -1.0, high: float = 1.0) -> np.ndarray:
    """Hard clip to prevent extreme gradient updates."""
    return np.clip(rewards, low, high)


def rank_rewards(rewards: np.ndarray) -> np.ndarray:
    """Replace rewards with their within-batch rank (anti-gaming)."""
    from scipy.stats import rankdata
    try:
        return rankdata(rewards) / len(rewards)
    except ImportError:
        # fallback without scipy
        order = np.argsort(rewards)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(len(rewards)) / (len(rewards) - 1)
        return ranks


norm_rewards = normalise_rewards(raw_rewards)
clipped      = clip_rewards(raw_rewards)
ranked       = rank_rewards(raw_rewards)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, vals, title, color in [
    (axes[0], norm_rewards, 'Z-score Normalised',    '#4a90d9'),
    (axes[1], clipped,      'Hard Clipped [-1, 1]',  '#7dc67d'),
    (axes[2], ranked,       'Rank-based (anti-game)', '#f5a623'),
]:
    ax.bar(range(len(vals)), vals, color=color)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(title); ax.set_xlabel('Sample index')
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('reward_engineering.png', dpi=120)
plt.show()

## 9. Summary — Domain-Specific Reward Design

| Domain | Reward source | Key insight |
|---|---|---|
| **Code** | Subprocess test execution | Binary but ground-truth; partial credit via test fractions |
| **Math** | Regex + numeric comparison | Must handle multiple answer formats; use tolerance |
| **Tool-use** | Environment simulation | Efficiency penalties prevent over-calling; error penalties prevent crashes |
| **Dialogue** | Heuristic metrics | Cheaper than human labels; weaker signal — use as auxiliary reward |

**Reward engineering tips:**
- **Normalise** rewards within each batch to prevent gradient scale issues
- **Clip** to prevent catastrophic updates from extreme outliers
- **Rank** instead of using raw values to reduce gaming (reward hacking)
- **Combine** signals: use a strong domain reward + a format/safety auxiliary reward
- **Audit regularly**: generate adversarial examples to stress-test your reward function
